In [3]:
import yfinance as yf
import pandas as pd
import os


StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 8, Finished, Available, Finished, False)

In [4]:
ticker = 'PKN.WA'
start_date = '2020-01-01'
source = 'yahoo_finance'

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 9, Finished, Available, Finished, False)

In [5]:
df = yf.download(
    tickers=ticker,
    start=start_date,
    interval='1d',
    auto_adjust=False,
    progress=False,
    multi_level_index=False
)

df

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 10, Finished, Available, Finished, False)

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2020-01-01,57.624134,85.820000,85.820000,85.820000,85.820000,0
2020-01-02,58.389595,86.959999,87.279999,85.699997,86.500000,644692
2020-01-03,57.556999,85.720001,86.639999,84.800003,86.500000,729205
2020-01-06,57.556999,85.720001,85.720001,85.720001,85.720001,0
2020-01-07,55.260624,82.300003,85.980003,82.160004,85.879997,1557711
...,...,...,...,...,...,...
2026-10-01,146.160004,146.160004,146.179993,144.000000,146.000000,1205649
2026-10-02,144.320007,144.320007,145.500000,141.199997,144.000000,1689904
2026-10-05,144.320007,144.320007,144.320007,144.320007,144.320007,0


In [6]:
if df.empty:
    raise ValueError(f'No data returned for {ticker}')

df = df.reset_index()

df['ticker'] = ticker
df['source'] = source
df['ingested_at'] = pd.Timestamp.now('UTC')

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 11, Finished, Available, Finished, False)

In [7]:
df = df.rename(columns={
    'Date' : 'trade_date',
    'Adj Close' : 'adj_close',
    'Close' : 'close',
    'High' : 'high',
    'Low' : 'low',
    'Open' : 'open',
    'Volume' : 'volume',
})

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 12, Finished, Available, Finished, False)

In [8]:
df.dtypes

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 13, Finished, Available, Finished, False)

trade_date          datetime64[ns]
adj_close                  float64
close                      float64
high                       float64
low                        float64
open                       float64
volume                       int64
ticker                      object
source                      object
ingested_at    datetime64[us, UTC]
dtype: object

In [9]:
ingestion_date = pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d')

output_dir = (
    f'/lakehouse/default/Files/market/yahoo/{ticker}/ingestion_date={ingestion_date}'
)

os.makedirs(output_dir, exist_ok=True)

output_path = f'{output_dir}/market_data.parquet'

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 14, Finished, Available, Finished, False)

In [10]:
df.to_parquet(output_path,index=False)

print(output_path)

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 15, Finished, Available, Finished, False)

/lakehouse/default/Files/market/yahoo/PKN.WA/ingestion_date=2026-10-07/market_data.parquet


In [11]:
df_check = pd.read_parquet(output_path)

df_check.head()

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 16, Finished, Available, Finished, False)

,trade_date,adj_close,close,high,low,open,volume,ticker,source,ingested_at
0,2020-01-01,57.624134,85.820000,85.820000,85.820000,85.820000,0,PKN.WA,yahoo_finance,2026-10-07 09:45:11.864413+00:00
1,2020-01-02,58.389595,86.959999,87.279999,85.699997,86.500000,644692,PKN.WA,yahoo_finance,2026-10-07 09:45:11.864413+00:00
2,2020-01-03,57.556999,85.720001,86.639999,84.800003,86.500000,729205,PKN.WA,yahoo_finance,2026-10-07 09:45:11.864413+00:00
3,2020-01-06,57.556999,85.720001,85.720001,85.720001,85.720001,0,PKN.WA,yahoo_finance,2026-10-07 09:45:11.864413+00:00
4,2020-01-07,55.260624,82.300003,85.980003,82.160004,85.879997,1557711,PKN.WA,yahoo_finance,2026-10-07 09:45:11.864413+00:00


In [12]:
print(f'Rows saved: {len(df_check)}')
print(f'From: {df_check["trade_date"].min()}')
print(f'To: {df_check["trade_date"].max()}')

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 17, Finished, Available, Finished, False)

Rows saved: 1703
From: 2020-01-01 00:00:00
To: 2026-10-07 00:00:00


In [13]:
assert len(df_check) > 0
assert df_check['ticker'].eq(ticker).all()
assert df_check['trade_date'].notna().all()

print('Bronze ingestion successful')

StatementMeta(, e9e83889-7a25-4a79-86bc-fdfe56309850, 19, Finished, Available, Finished, False)

Bronze ingestion successful
